In [56]:
import os
import pandas as pd
import numpy as np
from lets_plot import *
LetsPlot.setup_html()

# the pandas code to read all the CSV files into a single DataFrame

In [3]:
# Identify the location of the original files
# This represents the path: ../data/waitrose-2024-07
data_folder = os.path.join('..', 'data', 'waitrose-2024-07') #python way of doing the termiinal way

# Use a list comprehension to get all the files in the folder
all_files = [os.path.join(data_folder, file) for file in os.listdir(data_folder)
             if file.endswith('.csv')]

# Print the list of files if you want to check
# print(all_files)

# Read every single file as a DataFrame
# Save the dataframes in a list
list_of_dfs = [pd.read_csv(file) for file in all_files]

# Use pd.concat to concatenate all the files into a single DataFrame
df = pd.concat(list_of_dfs)

# Check that we have all the data
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25418 entries, 0 to 805
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   data-product-id        25418 non-null  int64  
 1   data-product-name      25418 non-null  object 
 2   data-product-type      25418 non-null  object 
 3   data-product-on-offer  25418 non-null  bool   
 4   data-product-index     25408 non-null  float64
 5   image-url              25418 non-null  object 
 6   product-page           25418 non-null  object 
 7   product-name           25407 non-null  object 
 8   product-size           25363 non-null  object 
 9   item-price             25407 non-null  object 
 10  price-per-unit         24976 non-null  object 
 11  offer-description      7201 non-null   object 
 12  category               25418 non-null  object 
dtypes: bool(1), float64(1), int64(1), object(10)
memory usage: 2.5+ MB


# 1. Create a new heading called ‘Exploring the item-price column’.

## Exploring the item-price column

In [9]:
df['item-price'] 

0       £3.50
1       £3.50
2       £9.00
3      £12.00
4       £3.75
        ...  
801     £7.00
802     £6.00
803     £1.05
804    £15.49
805    £14.99
Name: item-price, Length: 25418, dtype: object

many symbols, so cannot convert. need to clean

# 2. Data Cleaning

In [53]:
def clean_item_price(item_price: str):   
    if '£' in item_price:
        item_price = item_price.replace('£', '').strip()
    if 'p' in item_price:
        item_price = item_price.replace('p', '').strip()
    if 'each est.' in item_price:
        item_price = item_price.replace('each est.', '').strip()
    if "-" in item_price:
        low, high = item_price.split('-')
        item_price = (float(low) + float(high))/2



    return float(item_price)

df = df.dropna()
df['item-price'] = df['item-price'].apply(clean_item_price)


In [54]:
df

,data-product-id,data-product-name,data-product-type,data-product-on-offer,data-product-index,image-url,product-page,product-name,product-size,item-price,price-per-unit,offer-description,category
0,638635,Febreze Bathroom Air Freshener Cotton,G,True,1.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/febreze...,Febreze Bathroom Air Freshener Cotton,7.5ml,3.50,£46.67/100ml,Add 2 for £5,Home
1,440894,Febreze Air Mist Cotton Fresh,G,True,2.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/febreze...,Febreze Air Mist Cotton Fresh,185ml,3.50,£1.90/100ml,Add 2 for £5,Home
6,949851,Sodastream TERRA Starter Kit - Black,G,True,7.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/sodastr...,Sodastream TERRA Starter Kit - Black,each,59.99,£59.99 each,save £50.00. Was £109.99,Home
16,855897,John Lewis The Pan Frypan 28cm,G,True,17.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/john-le...,John Lewis The Pan Frypan 28cm,each,27.20,£27.20 each,20% Off. Was £34.00,Home
17,854837,John Lewis The Pan Omelette Pan,G,True,18.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/john-le...,John Lewis The Pan Omelette Pan,each,15.20,£15.20 each,20% Off. Was £19.00,Home
...,...,...,...,...,...,...,...,...,...,...,...,...,...
781,273716,Kleenex Easy Breathe Box Tissues,G,True,782.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/kleenex...,Kleenex Easy Breathe Box Tissues,56Sheet,1.75,£3.13/100 sheets,Introductory Offer.Will be £2.25,New
785,269504,Rock Face Face Wash,G,True,786.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/rock-fa...,Rock Face Face Wash,100ml,3.65,£3.65/100ml,Save 1/3. Was £5.50,New
786,269417,Fourpure Juiced Mango & Raspberry,G,True,787.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/fourpur...,Fourpure Juiced Mango & Raspberry,440ml,2.50,£5.69/litre,Mix & Match Add 5 for 4 Cheapest Item Free,New
800,202033,Willie's Cacao Chef's Drops 55% Sambirano Dark...,G,True,801.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/willies...,Willie's Cacao Chef's Drops 55% Sambirano Dark...,150g,3.30,£22/kg,Introductory Offer.Will be £4.40,New


# 3. 📊 Plotting the data

In [57]:
plot_df = (
    df.groupby('category')['item-price'].describe()
        .reset_index()
        .rename(columns={'25%': 'Q1', '50%': 'median', '75%': 'Q3'})
        .sort_values(by='median')
)

# plot_df.head() to see how it looks like

# This configures what shows up when you hover your mouse over the plot.
tooltip_setup = (
    layer_tooltips()
        .line('@category')
        .line('[@Q1 -- @median -- @Q3]')
        .format('@Q1', '£ {.2f}')
        .format('@median', '£ {.2f}')
        .format('@Q3', '£ {.2f}')
)

g = (
    # Maps the columns to the aesthetics of the plot.
    ggplot(plot_df, aes(y='category', x='median', xmin='Q1', xmax='Q3', fill='category')) +

    # GEOMS

    # Add a line range that 'listens to' columns informed in `ymin` and `ymax` aesthetics
    geom_linerange(size=1, alpha=0.75, tooltips=tooltip_setup) +

    # Add points to the plot (listen to `x` and `y` and fill aesthetics)
    geom_point(size=3, stroke=1, shape=21, tooltips=tooltip_setup) +

    # SCALES

    # Remove the legend (we can already read the categories from the y-axis)
    scale_fill_discrete(guide='none') +

    # Specify names for the axes
    scale_y_continuous(name="Categories\n(from largest to smallest median)", expand=[0.05, 0.05]) +
    scale_x_continuous(name="Price (£)", expand=[0., 0.05], format='£ {.2f}', breaks=np.arange(0, 20, 2.5)) +

    # LABELS
    # It's nice when the plot tells you the key takeaways
    labs(title='"Beer, Wine & Spirits" has the highest median price for individual items',
         subtitle="Dots represent the median price, bars represent the 25th and 75th percentiles") +
    theme(axis_text_x=element_text(size=15),
        axis_text_y=element_text(size=17),
        axis_title_x=element_text(size=20),
        axis_title_y=element_text(size=20),
        plot_title=element_text(size=19, face='bold'),
        plot_subtitle=element_text(size=18),
        legend_position='none') +
    ggsize(1000, 500)

)

g